In [6]:
#import libraries
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns
import time

print("Libraries loaded sucessfully")

Libraries loaded sucessfully


In [7]:
#load data
X_train = pd.read_csv('../data/processed/X_train_clean.csv')
X_val = pd.read_csv('../data/processed/X_val_clean.csv')
y_train = pd.read_csv('../data/processed/y_train_clean.csv').squeeze()
y_val = pd.read_csv('../data/processed/y_val_clean.csv').squeeze()

#enocde label to numbers
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_val_encoded = le.transform(y_val)

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape: {X_val.shape}")
print(f"\nLabel encoding mapping: ")
for i, label in enumerate(le.classes_):
    print(f" {label} -> {i}")
print("\nData Loaded Sucessfully")

X_train shape: (1980568, 71)
X_val shape: (423051, 71)

Label encoding mapping: 
 BENIGN -> 0
 Bot -> 1
 DDoS -> 2
 DoS GoldenEye -> 3
 DoS Hulk -> 4
 DoS Slowhttptest -> 5
 DoS slowloris -> 6
 FTP-Patator -> 7
 Heartbleed -> 8
 Infiltration -> 9
 PortScan -> 10
 SSH-Patator -> 11
 Web Attack - Brute Force -> 12
 Web Attack - Sql Injection -> 13
 Web Attack - XSS -> 14

Data Loaded Sucessfully


In [8]:
print("XGBoost Training")

start_time = time.time()

xgb_model = XGBClassifier(
    n_estimators = 100,
    max_depth=6,
    learning_rate= 0.1,
    random_state= 42,
    n_jobs= -1,
    eval_metric= 'mlogloss', # multiclass log loss -> way of measuring how wrong the model's predictions are across multiple classes
    verbosity= 1
)

xgb_model.fit(X_train, y_train_encoded)
training_time= time.time() - start_time
print(f"\nTrainign time completed in {training_time:.2f} seconds")

XGBoost Training

Trainign time completed in 99.46 seconds


In [9]:
#evaluate 

y_pred_encoded = xgb_model.predict(X_val)

#convert num back to labels
y_pred = le.inverse_transform(y_pred_encoded)
y_val_labels = le.inverse_transform(y_val_encoded)

print("\nClassification Report: ")
print(classification_report(y_val_labels, y_pred, digits=4))


Classification Report: 
                            precision    recall  f1-score   support

                    BENIGN     0.9996    0.9993    0.9995    339790
                       Bot     0.9655    0.6689    0.7903       293
                      DDoS     0.9998    0.9994    0.9996     19153
             DoS GoldenEye     0.9967    0.9948    0.9958      1540
                  DoS Hulk     0.9982    0.9998    0.9990     34427
          DoS Slowhttptest     0.9927    0.9878    0.9903       823
             DoS slowloris     0.9942    0.9942    0.9942       867
               FTP-Patator     0.9992    0.9992    0.9992      1187
                Heartbleed     1.0000    0.5000    0.6667         2
              Infiltration     1.0000    0.6000    0.7500         5
                  PortScan     0.9941    0.9997    0.9969     23757
               SSH-Patator     1.0000    1.0000    1.0000       882
  Web Attack - Brute Force     0.7101    0.9689    0.8195       225
Web Attack - Sql Injec

In [10]:
import joblib
joblib.dump(xgb_model, '../models/xgboost_clean.pkl')
print("Model saved.")

Model saved.


## XGBoost Results — Retrained on Clean 71-Column Data

### Overall Accuracy
XGBoost: 99.89% vs Random Forest: 99.59% — XGBoost wins on overall accuracy, but overall accuracy is misleading.

### Head to Head Comparison — Key Classes

| Class | RF Recall | XGB Recall | Verdict |
|-------|-----------|------------|---------|
| SQL Injection | 0.00 | 0.67 | XGBoost wins |
| Web Attack XSS | 0.56 | 0.12 | RF wins |
| Bot | 0.98 | 0.67 | RF wins |
| Heartbleed | 1.00 | 0.50 | RF wins |
| Brute Force | 0.73 | 0.97 | XGBoost wins |

### Core Insight
Fixing one rare class destroys the score of another. XGBoost improved SQL Injection from 0.00 to 0.67 but dropped XSS from 0.56 to 0.12. These models fail to handle class imbalance consistently across all minority classes simultaneously.

### Why IAA-Transformer is Needed
If this is how IDS systems work, it is a serious security problem. A model that misclassifies rare attack traffic as BENIGN lets real attacks through undetected. IAA-Transformer bakes class rarity directly into the attention mechanism, forcing consistent attention to ALL rare classes at the same time — not just some.